# Explorative Data Analysis

In [1]:
import json
import pandas as pd
import re

## Active Ads

In [2]:
with open("../data/raw_active_ads_2511.json", encoding="utf-8") as f:
    raw_active_df = pd.DataFrame(json.load(f))

raw_active_df.head()

,id,title,employer,municipality,region,occupation,published,deadline,description
0,30523189,PLM Teamcenter,ITMOTIVE in Sweden Aktiebolag,NaN,NaN,"Verksamhetskonsult, IT",2026-01-27T01:04:50,2026-12-26T23:59:59,"PLM Teamcenter Developer, Solution Architect.\..."
1,30896666,Teamchef Data & Analytics till Dizparc,Bravura Sverige AB,Linköping,Östergötlands län,"Affärskonsult, IT",2026-04-13T14:45:16,2026-10-10T23:59:59,Dizparc i Linköping bygger sitt nästa affärsbe...
2,30896809,Senior BI-konsult – forma framtidens datadrivn...,A Hub AB,Karlstad,Värmlands län,"Verksamhetskonsult, IT",2026-04-13T14:58:59,2026-10-10T23:59:59,Vill du arbeta i en roll där du tar fullt ansv...
3,30903986,Till vår kund Invid söker vi Affärssystems kon...,FollowU AB,Jönköping,Jönköpings län,"Verksamhetskonsult, IT",2026-04-14T15:38:38,2026-10-11T23:59:59,Business Central-konsulter till INVID\nVill du...
4,30915541,IT-Specialist Windows Server,Techrytera AB,Stockholm,Stockholms län,IT-arkitekt/Lösningsarkitekt,2026-04-16T14:50:43,2026-10-13T23:59:59,Om oss På Techrytera AB börjar rekryteringen m...


In [3]:
# Shape, cols, len
print(raw_active_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 289 entries, 0 to 288
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   id            289 non-null    str  
 1   title         289 non-null    str  
 2   employer      289 non-null    str  
 3   municipality  276 non-null    str  
 4   region        276 non-null    str  
 5   occupation    289 non-null    str  
 6   published     289 non-null    str  
 7   deadline      289 non-null    str  
 8   description   289 non-null    str  
dtypes: str(9)
memory usage: 20.4 KB
None


In [4]:
# nans

missing = (
    raw_active_df.isna() |
    raw_active_df.astype(str).apply(lambda col: col.str.strip().eq(""))
).sum()

missing_summary = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": (missing / len(raw_active_df) * 100).round(2)
}).sort_values("missing_pct", ascending=False)

print(missing_summary)

              missing_count  missing_pct
region                   13          4.5
municipality             13          4.5
id                        0          0.0
employer                  0          0.0
title                     0          0.0
occupation                0          0.0
published                 0          0.0
deadline                  0          0.0
description               0          0.0


In [5]:
# duplicates

print("Duplicated rows based on id:", raw_active_df["id"].duplicated().sum())


# duplicates based on title + employer + description
duplicate_ads = raw_active_df.duplicated(
    subset=["title", "employer", "description"]
).sum()

print("\nDuplicates based on title + employer + description:", duplicate_ads) 


duplicate_texts = raw_active_df[raw_active_df["description"].duplicated(keep=False)]

duplicate_texts[["id","title","employer","published","description"]].head(10)

Duplicated rows based on id: 0

Duplicates based on title + employer + description: 6


,id,title,employer,published,description
14,30981932,Business Consultant M3 - Usability/Extensibili...,Columbus Sweden AB,2026-04-30T15:05:34,Vi söker en tekniskt stark konsult med fokus p...
15,30981944,Business Consultant M3 - Usability/Extensibili...,Columbus Sweden AB,2026-04-30T15:06:16,Vi söker en tekniskt stark konsult med fokus p...
16,30981973,Business Consultant M3 - Usability/Extensibili...,Columbus Sweden AB,2026-04-30T15:07:14,Vi söker en tekniskt stark konsult med fokus p...
17,30990931,Data Engineer - Stockholm,Columbus Sweden AB,2026-05-04T15:46:48,Arbetsuppgifter\n\nSom Data Engineer på Columb...
18,30990940,Data Engineer - Göteborg,Columbus Sweden AB,2026-05-04T15:47:29,Arbetsuppgifter\n\nSom Data Engineer på Columb...
19,30990944,Data Engineer - Malmö,Columbus Sweden AB,2026-05-04T15:48:16,Arbetsuppgifter\n\nSom Data Engineer på Columb...
43,31222965,Integrations- och Lösningsarkitekt –,Adavo AB,2026-06-26T16:39:56,Startdatum: September Slutdatum: Option på för...
44,31222967,Integrations- och Lösningsarkitekt –,Adavo AB,2026-06-26T16:40:00,Startdatum: September Slutdatum: Option på för...
105,31468539,Technical Business Analyst,Cognizant Technology Solutions Sweden AB,2026-09-11T15:32:05,What makes Cognizant a unique place to work? T...
106,31468610,Technical Business Analyst,Cognizant Technology Solutions Sweden AB,2026-09-11T15:32:50,What makes Cognizant a unique place to work? T...


In [6]:
# distribution of titles
unique_titles = raw_active_df["title"].nunique()

print(f"Of {len(raw_active_df)} ads, the unique titles are {unique_titles}.\n")
print(raw_active_df["title"].value_counts().head(10))

Of 289 ads, the unique titles are 259.

title
Kravanalytiker                                        10
Systemarkitekt                                         4
IT-arkitekt                                            4
Senior Enterprise Arkitekt - Business Intelligence     4
Lösningsarkitekt                                       3
Technical Business Analyst                             3
Senior Kravanalytiker                                  2
Integrations- och Lösningsarkitekt –                   2
Mjukvaruarkitekter                                     2
Software Architect                                     2
Name: count, dtype: int64


In [7]:
# Occupatioal group according to SSYK
unique_occupations = raw_active_df["occupation"].nunique()

print(f"Total unique occupations: {unique_occupations}\n")
print(raw_active_df["occupation"].value_counts().head(10))

Total unique occupations: 18

occupation
IT-arkitekt/Lösningsarkitekt       64
Kravanalytiker                     40
Affärskonsult, IT                  31
Systemarkitekt                     28
Enterprise Architect               21
Data scientist                     18
Applikationskonsult                16
Systemanalytiker/Systemutredare    15
IT-arkitekt                        14
Verksamhetskonsult, IT             13
Name: count, dtype: int64


In [8]:
# geographical distribution
print("Nmber of municipalities:", raw_active_df["municipality"].nunique())
print(raw_active_df["municipality"].value_counts().head(10))

print("\nNmber of regions:", raw_active_df["region"].nunique())
print(raw_active_df["region"].value_counts().head(10))

Nmber of municipalities: 44
municipality
Stockholm     110
Göteborg       38
Linköping      15
Malmö          15
Lund            9
Solna           8
Sundbyberg      7
Norrköping      6
Jönköping       5
Örebro          5
Name: count, dtype: int64

Nmber of regions: 19
region
Stockholms län          135
Västra Götalands län     45
Skåne län                27
Östergötlands län        24
Västmanlands län          7
Jönköpings län            5
Örebro län                5
Uppsala län               4
Hallands län              4
Dalarnas län              3
Name: count, dtype: int64


In [9]:
# date distribution
raw_active_df["published"] = pd.to_datetime(raw_active_df["published"])

raw_active_df["year_month"] = raw_active_df["published"].dt.to_period("M")
monthly_ads = raw_active_df["year_month"].value_counts().sort_index()

print(monthly_ads)

year_month
2026-01      1
2026-04     16
2026-05     11
2026-06     18
2026-07     12
2026-08     20
2026-09    158
2026-10     53
Freq: M, Name: count, dtype: int64


In [10]:
# description 
raw_active_df["description_length"] = raw_active_df["description"].str.len()
print("Description lenght:")
print(raw_active_df["description_length"].describe())

print(
    "\nVery short descriptions < 200:",
    (raw_active_df["description_length"] < 200).sum()
)

Description lenght:
count     289.000000
mean     3924.795848
std      1231.171271
min       146.000000
25%      3104.000000
50%      3876.000000
75%      4773.000000
max      6469.000000
Name: description_length, dtype: float64

Very short descriptions < 200: 1


## Historic Ads

In [11]:
with open("../data/raw_historic_ads_2511.json", encoding="utf-8") as f:
    raw_historic_df = pd.DataFrame(json.load(f))

## Taxonomi

In [12]:
with open("../data/taxonomi_2511.json", encoding="utf-8") as f:
    taxonomi_df = pd.DataFrame(json.load(f))